# RevB 测试结果后处理

输入一个 **task-id**，自动处理 `noise` 或 `offset_drift`，在此查看摘要和图表。
输入来自 `tasks/<task-id>/`，结果写回同一任务的 `results/`；同时刷新全任务总览。

1. 修改下面的参数单元。
2. 选择 **Run → Run All Cells**。任务列表会在处理前显示。
3. 查看采集状态、数据完整性和结果。再次更换任务或参数时，重新运行全部单元。

`REGENERATE_SPECTRA=False`：复用已有 NPZ，缺失的频谱从 TDMS 补算。
`True`：从原始 TDMS 重新生成该 noise 任务的频谱并覆盖已有 NPZ。
该选项只对 noise 生效；offset drift 每次重新分析有效温度阶段。

RevA 及更早的历史文件继续归档，不在本 Notebook 的处理范围内。
首次使用见同目录 [README](README.md)。

In [ ]:
# 主要修改这两项。task-id 使用任务目录名；可在下方任务列表中查找。
TASK_ID = "RevB_Chip_1_wiShield_Gmx_default_OSC_default_Vsup_pm6V"
REGENERATE_SPECTRA = False

# 高级选项：默认即可。换电脑后若数据在其他磁盘，可填绝对路径。
# Windows 示例：TASKS_ROOT = r"D:\Unicorn_Testbench\Unicorn_CSA_RevB-COB\tasks"
TASKS_ROOT = None
WELCH_NPERSEG = 262_144  # 仅影响新生成的频谱；修改算法参数时请选择全部重算。

## 任务列表与输入条件

路径从当前工作目录及其上级自动定位。请从仓库或 `postprocess/` 目录启动 Jupyter。
复制任务到其他电脑时保留完整 `data/`、`spectra/`、`provenance/` 和 `task.json`。

In [ ]:
from contextlib import redirect_stdout
from html import escape
import importlib.util
import io
from pathlib import Path
import sys

from IPython.display import Image, Markdown, display


def find_postprocess_directory():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base / "Unicorn_CSA_RevB-COB" / "postprocess", base / "postprocess", base):
            if ((candidate / "process_tasks.py").is_file()
                    and (candidate.parent / "analyze_dut_revb_pmu_noise.py").is_file()):
                return candidate.resolve()
    raise FileNotFoundError("未找到 RevB 后处理代码，请从仓库或 postprocess 目录启动 Jupyter。")


def show_table(headers, rows):
    def text(value):
        return escape(str(value)).replace("|", "&#124;").replace("\n", "<br>")
    lines = ["| " + " | ".join(map(text, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines.extend("| " + " | ".join(map(text, row)) + " |" for row in rows)
    display(Markdown("\n".join(lines)))


def number(value, scale=1.0):
    return "—" if value is None else f"{value * scale:.8g}"


postprocess_dir = find_postprocess_directory()
spec = importlib.util.spec_from_file_location("revb_notebook_processing", postprocess_dir / "process_tasks.py")
replay = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = replay
spec.loader.exec_module(replay)
tasks_root = (Path(TASKS_ROOT).expanduser() if TASKS_ROOT is not None
              else postprocess_dir.parent / "tasks").resolve()
if not tasks_root.is_dir():
    raise FileNotFoundError(f"任务目录不存在：{tasks_root}")

available = []
for manifest in sorted(tasks_root.glob("*/task.json")):
    entry = replay.load_task(manifest.parent)
    if entry["measurement_type"] in {"noise", "offset_drift"}:
        available.append([entry["task_id"], entry["measurement_type"], entry.get("chip", "—"),
                          entry.get("acquisition_status", "unknown")])
print(f"任务目录：{tasks_root}")
show_table(["task-id", "类型", "Chip", "采集状态"], available)

In [ ]:
# 清空上次运行的展示状态，避免失败后误看旧任务结果。
status = None
summary = None
figure_paths = []

if not isinstance(TASK_ID, str) or not TASK_ID.strip():
    raise ValueError("请在参数单元填写 TASK_ID，再重新运行全部单元。")
if type(REGENERATE_SPECTRA) is not bool:
    raise TypeError("REGENERATE_SPECTRA 必须为 True 或 False。")
task_id = replay.task_id_value(TASK_ID.strip())
task_dir = (tasks_root / task_id).resolve()
if task_dir.parent != tasks_root:
    raise ValueError("任务目录必须位于所选 tasks 目录中。")
if not (task_dir / "task.json").is_file():
    raise FileNotFoundError(f"没有找到该 task-id 的 task.json：{task_id}")
task = replay.load_task(task_dir)
measurement_type = task["measurement_type"]
if measurement_type not in {"noise", "offset_drift"}:
    raise ValueError(f"本 Notebook 处理 noise 和 offset_drift，当前类型为 {measurement_type}。")
if replay.acquisition_in_progress(task_dir, task):
    raise replay.AcquisitionInProgressError("该任务还在采集，请使用已完成采集的任务副本。")
if measurement_type == "noise" and (type(WELCH_NPERSEG) is not int or WELCH_NPERSEG < 2):
    raise ValueError("WELCH_NPERSEG 必须是大于等于 2 的整数。")

show_table(["输入条件", "值"], [
    ["task-id", task_id], ["类型", measurement_type], ["Chip", task.get("chip", "—")],
    ["采集状态", task.get("acquisition_status", "unknown")],
    ["增益 V/V", task.get("gain_v_per_v", "由原始数据读取")],
    ["频谱选项", ("从 TDMS 全部重算" if REGENERATE_SPECTRA else "复用已有，缺失补算")
                    if measurement_type == "noise" else "不适用；重算有效温度阶段的 offset drift"],
    ["结果目录", task_dir / "results"],
])

# 全部重算必须有对应原始波形，防止缺少 TDMS 时静默沿用旧频谱。
if measurement_type == "noise" and REGENERATE_SPECTRA:
    analyzer = replay.legacy_module("analyze_dut_revb_pmu_noise.py")
    plotter = replay.legacy_module("plot_dut_revb_pmu_noise.py")
    raw_runs = {run for run, _ in analyzer.numbered_tdms_files(task_dir / "data", task_id)}
    cached_runs = {run for run, _ in plotter.numbered_spectrum_files(task_dir / "spectra", task_id)}
    if not raw_runs or cached_runs - raw_runs:
        raise FileNotFoundError("重新生成 spectra 需要相应的原始 TDMS；请补齐 data/ 后重试。")

## 执行后处理

沿用现有 Welch、PSD 平均及 DMM 分阶段算法，核对原始文件哈希和任务/阶段归属。
采集未完成、数据不完整和处理失败会分别显示；数据完整不等于器件规格通过。

In [ ]:
status = None
summary = None
figure_paths = []
processing_log = io.StringIO()
try:
    with redirect_stdout(processing_log):
        new_status = replay.process_task(
            task_dir,
            recompute_spectra=REGENERATE_SPECTRA if measurement_type == "noise" else False,
            nperseg=WELCH_NPERSEG,
        )
        new_summary = replay.read_json(task_dir / new_status["summary_file"])
        replay.write_overview(tasks_root, tasks_root.parent / "results")
except Exception:
    print(processing_log.getvalue())
    raise
else:
    status, summary = new_status, new_summary

show_table(["处理状态", "值"], [
    ["采集状态", status["acquisition_status"]],
    ["数据状态", status["data_status"]],
    ["后处理状态", status["processing_status"]],
    ["原始 TDMS 数", status["raw_file_count"]],
])
for note in status.get("notes", []):
    print(f"说明：{note}")
print(f"结果已写入：{task_dir / 'results'}")
print("详细计算日志保存在本次 Notebook 会话的 processing_log.getvalue() 中。")

## 结果摘要与图表

Noise 单位为输入等效 `nV/√Hz`。Offset drift 展示各阶段均值、样本标准差和控温实测范围；
斜率按名义温度设定值拟合，只作为这次测试的描述量。9 月 18 日的控温稳定性问题仍需结合采集记录判断。

In [ ]:
if status is None or summary is None:
    raise RuntimeError("请先成功执行该任务的后处理单元。")

if measurement_type == "noise":
    show_table(["Noise 指标", "值"], [
        ["频谱文件数", status["spectrum_file_count"]],
        ["本次新生成/重算", status["spectra_created"]],
        ["本次复用", status["spectra_reused"]],
        ["1 Hz 输入等效噪声 nV/√Hz", number(status["noise_1_hz_nv_per_sqrt_hz"])],
        ["1 kHz 输入等效噪声 nV/√Hz", number(status["noise_1000_hz_nv_per_sqrt_hz"])],
    ])
    figure_paths = [task_dir / "results" / f"{task_id}_average_input_noise_asd.png",
                    task_dir / "results" / f"{task_id}_target_asd_distributions.png"]
else:
    show_table(["Offset drift 指标", "值"], [
        ["有效读数", status["valid_reading_count"]],
        ["有效温度阶段", status["offset_stage_count"]],
        ["名义温度拟合斜率 µV/°C", number(status["nominal_temperature_linear_slope_uv_per_c"])],
    ])
    if summary["points"]:
        show_table(["阶段", "设定温度 °C", "实测温度均值 °C", "实测范围 °C", "读数",
                    "输入 offset 均值 µV", "样本标准差 µV", "温度超带样本数"], [
            [point["stage_index"], point["target_c"], number(point["product_temperature_mean_c"]),
             f"{number(point['product_temperature_min_c'])} ~ {number(point['product_temperature_max_c'])}",
             point["reading_count"], number(point["input_offset_mean_v"], 1e6),
             number(point["input_offset_sample_std_v"], 1e6), point["temperature_out_of_band_samples"]]
            for point in summary["points"]
        ])
        figure_paths = [task_dir / "results" / "offset_vs_temperature.png"]
    else:
        display(Markdown("**没有有效 offset 读数，因此不生成温漂数值或 offset 图。**"))
    for stage in summary["stages"]:
        if stage.get("error"):
            print(f"阶段 {stage['stage_index']}：{stage['error']}")
    # 采集时保存的控温曲线只用于查看当时温度变化，不当作新的温漂拟合。
    temperature_plot = task_dir / "results" / "temperature_curve.png"
    if temperature_plot.is_file():
        figure_paths.append(temperature_plot)

for path in figure_paths:
    if path.is_file():
        display(Markdown(f"**{path.name}**"))
        display(Image(filename=str(path)))
    else:
        print(f"图表未生成：{path.name}")

## 输出文件

任务结果：`tasks/<task-id>/results/`。Noise 频谱：同任务的 `spectra/`。
任务摘要：`results/processing_status.json`。全任务总览：`tasks/` 同级的 `results/task_overview.csv` 和 `.json`。

原始 `data/` 和采集 `provenance/` 不改写。Notebook 提交时保留空输出，避免把图表和本机路径重复嵌入文件。